# Tập 3.3 · EKF và UKF: Kalman cho thế giới cong 🌀

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap33/tap33_ekf_ukf.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~45 phút · Giáo trình: Principles of Robot Autonomy, [Chương 12](https://porabook.com/book/approximate-filters/) (Mục 12.3)

EKF xấp xỉ đường cong bằng đường thẳng; UKF bắn 7 điểm sigma qua hàm cong thật. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (Phần A: một bước cong, Phần B: định vị 60 s).
2. **Tự viết** hai ma trận Jacobi của EKF và các điểm sigma của UKF (3 bài), có ô chấm tự động.
3. Vẽ elip EKF/UKF bằng hàm của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap33")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import ekf_ukf as goc              # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from kiem_tra import rms
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"Phần A: Monte Carlo tâm x = {goc.MC_MU[0]:.3f} · EKF {goc.EKF_MU[0]:.3f} · UKF {goc.UKF_MU[0]:.3f} m")
L = goc.LOG_B
print(f"Phần B: chỉ odometry RMS {rms(L, 'dr') * 100:.1f} cm · EKF {rms(L, 'ekf') * 100:.1f} cm · UKF {rms(L, 'ukf') * 100:.1f} cm")

## 2. Bài tập

$f(x,u) = [p_x + v\,dt\cos\theta,\ p_y + v\,dt\sin\theta,\ \theta + \omega\,dt]$, $\quad h(x,\ell) = [\sqrt{\Delta x^2 + \Delta y^2},\ \operatorname{atan2}(\Delta y, \Delta x) - \theta]$

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `F_jac(x, u, dt)` | $F = \partial f/\partial x$ (3×3) |
| **2** | `H_jac(x, l)` | $H = \partial h/\partial x$ (2×3) |
| **3** | `diem_sigma(mu, Sig)` | $\lambda = \alpha^2(n+\kappa) - n$, 7 điểm $\mu \pm$ cột của $\sqrt{(n+\lambda)\Sigma}$, trọng số $W_m, W_c$ |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

ALPHA, BETA, KAPPA = 1.0, 2.0, 0.0


def F_jac(x, u, dt):
    """Bài 1 (EKF): ma trận Jacobi F = ∂f/∂x (3×3) tại x. Chỉ θ làm vị trí đổi phi tuyến:

        F = [[1, 0, −v·dt·sin θ],
             [0, 1,  v·dt·cos θ],
             [0, 0,  1         ]]"""
    raise NotImplementedError("TODO Bài 1: Jacobi của chuyển động")


def H_jac(x, l):
    """Bài 2 (EKF): ma trận Jacobi H = ∂h/∂x (2×3) của phép đo khoảng cách + góc tới mốc l.

    Với Δx = l_x − p_x, Δy = l_y − p_y, q = Δx² + Δy², r = √q:
        H = [[−Δx/r, −Δy/r,  0],
             [ Δy/q, −Δx/q, −1]]"""
    raise NotImplementedError("TODO Bài 2: Jacobi của phép đo")


def diem_sigma(mu, Sig):
    """Bài 3 (UKF): 2n + 1 điểm sigma và trọng số (n = len(mu)).

    λ = α²(n + κ) − n;  L = cholesky((n + λ)·Σ)  (np.linalg.cholesky, cột i là L[:, i])
    Điểm: X_0 = μ,  X_i = μ + L[:, i],  X_{n+i} = μ − L[:, i]   (i = 0..n−1, xếp đúng thứ tự này)
    Trọng số: Wm = Wc = 1 / (2(n + λ)) cho mọi điểm, trừ Wm[0] = λ/(n + λ), Wc[0] = λ/(n + λ) + (1 − α² + β).
    Trả về (X dạng mảng (2n+1, n), Wm, Wc)."""
    raise NotImplementedError("TODO Bài 3: điểm sigma")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1 và 3, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Hướng càng rõ thì EKF càng ít sai? Đổi độ mơ hồ hướng ở Phần A và so EKF với Monte Carlo.

In [ ]:
rng = np.random.default_rng(0)
for do in [5, 15, 35, 60]:
    Sig = np.diag([0.05 ** 2, 0.05 ** 2, np.radians(do) ** 2])
    mc = np.array([goc.f(x, goc.U_A, goc.DT_A) for x in rng.multivariate_normal(goc.MU_A, Sig, 20000)])[:, :2].mean(0)
    X, Wm, _ = goc.diem_sigma(goc.MU_A, Sig)
    ukf = Wm @ np.array([goc.f(x, goc.U_A, goc.DT_A) for x in X])[:, :2]
    print(f"hướng ±{do:2d}°: Monte Carlo x = {mc[0]:.3f} · EKF 2.000 (lệch {abs(2 - mc[0]) * 100:4.1f} cm) · UKF {ukf[0]:.3f} (lệch {abs(ukf[0] - mc[0]) * 100:4.1f} cm)")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap33).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)